# ACT: chunks, CVAE и temporal ensemble

**Learning Fine-Grained Bimanual Manipulation with Low-Cost Hardware** · Zhao et al. · 2023

[Оригинальная статья](https://arxiv.org/abs/2304.13705)

**Цель урока:** Написать маленькую conditional VAE для chunks действий, отличить posterior training от inference и объединить перекрывающиеся predictions.

## Перед началом

Вводный VLA, action chunks, Gaussian latent и KL.

Около 40–60 минут с заданиями. CPU, PyTorch и Matplotlib; без скачивания весов и датасетов. Время — ориентир для чтения и работы, а не benchmark выполнения.

## Задача и идея

ACT предсказывает последовательность будущих действий вместо одного следующего действия. CVAE позволяет моделировать вариативность демонстраций: posterior во время обучения видит target actions, decoder использует latent вместе с наблюдением. На inference target actions недоступны.

Temporal ensembling объединяет предсказания для одного абсолютного момента, сделанные в разные моменты. Это не среднее произвольных позиций chunk. Здесь state-only MLP заменяет visual Transformer, чтобы детально проверить objective и индексацию на CPU.

## Граница воспроизведения

Учебная state-only CVAE и same-time ensembling. Не официальный ACT visual Transformer, не LeRobot trainer и не проверка реального робота.

Код написан для этого урока по математическим идеям статьи. Это не скопированный авторский репозиторий. Успех assertions подтверждает указанные свойства реализации, но не результаты статьи.

## Формула → тензоры

`z = mu + exp(0.5*logvar)*epsilon`; `KL(q||N(0,I)) = 0.5*sum(mu² + exp(logvar) - 1 - logvar)`.

`loss = masked_L1(actions_hat, actions) + beta*KL`. Reconstruction усредняется только по валидным координатам; KL — по batch после суммы latent dimensions. В этом примере inference берёт z=0. Ensemble ниже использует `weight=exp(-k*age)`, где age больше у более старого chunk: это явно выбранная recency-weighted convention, не утверждение о точной convention оригинального кода.

In [ ]:
%matplotlib inline
import math
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(42)
plt.rcParams.update({'figure.figsize': (7, 3), 'axes.spines.top': False, 'axes.spines.right': False})
print('PyTorch:', torch.__version__, '| CPU | seed=42')

## Шаг 1

Posterior получает state и demonstration chunk; inference decoder получает только state и z=0.

In [ ]:
B, K, A, Z = 48, 4, 2, 3
state = torch.randn(B, 3)
actions = state[:, None, :2] * 0.3 + torch.arange(K)[None, :, None] * 0.1
valid = torch.ones(B, K, dtype=torch.bool); valid[:12, -1] = False
class ChunkCVAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.posterior = nn.Sequential(nn.Linear(3 + K*A, 32), nn.ReLU(), nn.Linear(32, 2*Z))
        self.decoder = nn.Sequential(nn.Linear(3 + Z, 32), nn.ReLU(), nn.Linear(32, K*A))
    def decode(self, observation, z):
        return self.decoder(torch.cat([observation, z], -1)).reshape(-1, K, A)
    def forward(self, observation, targets, mask):
        # Zero padded targets before feeding the posterior as well as masking loss.
        encoded = torch.cat([observation, (targets * mask[..., None]).flatten(1)], -1)
        mu, logvar = self.posterior(encoded).chunk(2, -1)
        z = mu + (0.5 * logvar).exp() * torch.randn_like(mu)
        return self.decode(observation, z), mu, logvar
model = ChunkCVAE()
assert model.decode(state, torch.zeros(B, Z)).shape == actions.shape

## Шаг 2

Проверяем нулевой KL стандартного нормального posterior и обучаем objective с маской.

In [ ]:
def kl_normal(mu, logvar):
    return 0.5 * (mu.square() + logvar.exp() - 1 - logvar).sum(-1).mean()
assert kl_normal(torch.zeros(B, Z), torch.zeros(B, Z)) == 0
optimizer = torch.optim.AdamW(model.parameters(), lr=0.01)
history = []
for step in range(100):
    prediction, mu, logvar = model(state, actions, valid)
    reconstruction = ((prediction - actions).abs() * valid[..., None]).sum() / (valid.sum() * A)
    loss = reconstruction + 0.01 * kl_normal(mu, logvar)
    optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step(); history.append(loss.item())
with torch.no_grad():
    predicted = model.decode(state, torch.zeros(B, Z))
print('z=0 inference L1 on train:', (predicted - actions).abs().mean().item())
assert torch.isfinite(predicted).all()
plt.plot(history, color='#d73535'); plt.xlabel('step'); plt.ylabel('L1 + beta KL'); plt.show()

## Шаг 3

В момент t=2 доступны a[0,2], a[1,1], a[2,0]. Даже нулевой action — валидное предсказание, а не признак отсутствия данных.

In [ ]:
chunks = torch.tensor([[[0., 0.], [1., 1.], [2., 2.], [3., 3.]],
                       [[10., 10.], [11., 11.], [12., 12.], [13., 13.]],
                       [[20., 20.], [21., 21.], [22., 22.], [23., 23.]]])
now = 2; starts = torch.arange(3)
candidates = chunks[starts, now - starts]
ages = (now - starts).float(); weights = torch.exp(-0.5 * ages); weights /= weights.sum()
ensemble = (candidates * weights[:, None]).sum(0)
print('same-time candidates:', candidates.tolist(), 'weights:', weights.tolist(), 'ensemble:', ensemble.tolist())
assert torch.equal(candidates[:, 0], torch.tensor([2., 11., 20.]))
assert ensemble.shape == (A,) and torch.allclose(weights.sum(), torch.tensor(1.))

## Как читать результат

KL контролирует posterior, но не гарантирует полезный latent; слишком большой beta может вызвать posterior collapse. Низкий reconstruction loss с target-conditioned posterior не равен качеству z=0 inference. Ensemble требует явных timestamps и validity masks.

## Самостоятельные эксперименты

1. Увеличьте beta в 100 раз и сравните KL/reconstruction отдельно.
2. Подайте actions на inference: почему это leakage?
3. Удалите старый chunk и пересчитайте ensemble. Не используйте action==0 как validity.

Сначала запишите прогноз, затем меняйте по одному фактору. Для честного сравнения сбрасывайте ядро и выполняйте все ячейки с начала. Сохраните копию для своих изменений.

## Проверка рассуждений — после эксперимента

Большой beta прижимает posterior к prior и может лишить decoder информации latent. На deployment target actions неизвестны. Chunk validity определяется наличием предсказания и его временем, а не числовым значением action.

## Что записать в лабораторный журнал

- Гипотеза и изменённый параметр.
- Seed, размер данных, число шагов и используемая precision.
- Значения проверок до и после, график и объяснение расхождения с прогнозом.
- Что этот эксперимент **не** позволяет заключить.

Связанный раздел доступен по кнопке «Открыть раздел студии» над уроком.